# 02 — Preprocessing

Visualise before/after for each preprocessing step, R-peak detection, frequency spectra, and segment quality.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import signal as sp_signal

import config
from data_loader import load_ptbxl, load_mitbih
from preprocessor import (
    Preprocessor,
    bandpass_filter,
    notch_filter,
    remove_baseline_wander,
    resample_signal,
    z_score_normalize,
    pan_tompkins_rpeaks,
    segment_beats,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Load a sample signal

In [ ]:
try:
    ds = load_mitbih(config.PATHS.mitbih)
    # Get a raw single-channel signal — reload a full record
    import wfdb
    record_path = str(list(config.PATHS.mitbih.glob('*.hea'))[0].with_suffix(''))
    rec = wfdb.rdrecord(record_path, channels=[0])
    raw_sig = rec.p_signal[:, 0].astype('float32')
    FS = rec.fs
    print(f'Loaded record: {record_path}  fs={FS} Hz  length={len(raw_sig)} samples')
except Exception as e:
    print(f'[INFO] Using synthetic signal: {e}')
    FS = 360
    t = np.arange(FS * 10) / FS
    raw_sig = (
        np.sin(2 * np.pi * 1.0 * t)          # heartbeat
        + 0.3 * np.sin(2 * np.pi * 50.0 * t)  # powerline
        + 0.2 * np.sin(2 * np.pi * 0.1 * t)   # baseline drift
        + 0.05 * np.random.randn(len(t))       # noise
    ).astype('float32')

# Use a 10-second window
n_window = FS * 10
sig = raw_sig[:n_window]

## 2. Bandpass filter — before / after

In [ ]:
bp_sig = bandpass_filter(sig[np.newaxis, np.newaxis, :], 0.5, 40.0, FS)[0, 0]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                     subplot_titles=['Raw signal', 'After bandpass (0.5–40 Hz)'])
t = np.arange(len(sig)) / FS
fig.add_trace(go.Scatter(x=t.tolist(), y=sig.tolist(), name='raw', line=dict(width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=t.tolist(), y=bp_sig.tolist(), name='bandpass', line=dict(width=1, color='green')), row=2, col=1)
fig.update_layout(height=500, title='Bandpass Filter', template='plotly_white')
fig.show()

## 3. Notch filter — powerline removal

In [ ]:
notch_sig = notch_filter(sig[np.newaxis, np.newaxis, :], 50.0, FS)[0, 0]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                     subplot_titles=['Raw signal', 'After notch (50 Hz removed)'])
fig.add_trace(go.Scatter(x=t.tolist(), y=sig.tolist(), name='raw', line=dict(width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=t.tolist(), y=notch_sig.tolist(), name='notch', line=dict(width=1, color='purple')), row=2, col=1)
fig.update_layout(height=500, title='Notch Filter', template='plotly_white')
fig.show()

## 4. Frequency spectrum — FFT before/after filtering

In [ ]:
def _psd(s, fs):
    freqs, power = sp_signal.welch(s, fs=fs, nperseg=512)
    return freqs, 10 * np.log10(power + 1e-12)

f_raw, p_raw = _psd(sig, FS)
full_proc = bandpass_filter(notch_filter(sig[np.newaxis, np.newaxis, :], 50.0, FS), 0.5, 40.0, FS)[0, 0]
f_proc, p_proc = _psd(full_proc, FS)

fig = go.Figure()
fig.add_trace(go.Scatter(x=f_raw.tolist(), y=p_raw.tolist(), name='Raw PSD'))
fig.add_trace(go.Scatter(x=f_proc.tolist(), y=p_proc.tolist(), name='Filtered PSD'))
fig.update_layout(title='Power Spectral Density', xaxis_title='Frequency (Hz)',
                  yaxis_title='Power (dB)', template='plotly_white', xaxis_range=[0, 100])
fig.show()

## 5. Baseline wander removal

In [ ]:
# Add synthetic drift to a clean segment
t_arr = np.arange(len(bp_sig)) / FS
drift = (0.5 * np.sin(2 * np.pi * 0.05 * t_arr)).astype('float32')
drifted = bp_sig + drift
corrected = remove_baseline_wander(drifted[np.newaxis, np.newaxis, :])[0, 0]

fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                     subplot_titles=['With baseline drift', 'Detected baseline', 'After removal'])
fig.add_trace(go.Scatter(x=t_arr.tolist(), y=drifted.tolist(), name='drifted', line=dict(width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=t_arr.tolist(), y=drift.tolist(), name='drift', line=dict(width=1, color='red')), row=2, col=1)
fig.add_trace(go.Scatter(x=t_arr.tolist(), y=corrected.tolist(), name='corrected', line=dict(width=1, color='green')), row=3, col=1)
fig.update_layout(height=600, title='Baseline Wander Removal', template='plotly_white')
fig.show()

## 6. R-peak detection visualization

In [ ]:
rpeaks = pan_tompkins_rpeaks(full_proc, FS)
print(f'Detected {len(rpeaks)} R-peaks in {len(full_proc)/FS:.1f} s')
if len(rpeaks) > 1:
    rr_ms = np.diff(rpeaks) / FS * 1000
    print(f'RR intervals: mean={rr_ms.mean():.1f} ms  std={rr_ms.std():.1f} ms  HR≈{60000/rr_ms.mean():.0f} bpm')

fig = go.Figure()
t_arr = np.arange(len(full_proc)) / FS
fig.add_trace(go.Scatter(x=t_arr.tolist(), y=full_proc.tolist(), name='ECG', line=dict(width=1)))
if len(rpeaks) > 0:
    fig.add_trace(go.Scatter(
        x=(rpeaks / FS).tolist(), y=full_proc[rpeaks].tolist(),
        mode='markers', name='R-peaks', marker=dict(color='red', size=8, symbol='x'),
    ))
fig.update_layout(title='R-peak Detection (Pan-Tompkins)', xaxis_title='Time (s)',
                  template='plotly_white')
fig.show()

## 7. Resampling

In [ ]:
sig_250 = resample_signal(full_proc[np.newaxis, np.newaxis, :], orig_fs=FS, target_fs=250)[0, 0]
t_250 = np.arange(len(sig_250)) / 250

print(f'Original: {len(full_proc)} samples @ {FS} Hz = {len(full_proc)/FS:.2f} s')
print(f'Resampled: {len(sig_250)} samples @ 250 Hz = {len(sig_250)/250:.2f} s')

fig = make_subplots(rows=2, cols=1, subplot_titles=[f'Original ({FS} Hz)', 'Resampled (250 Hz)'])
fig.add_trace(go.Scatter(x=t_arr.tolist(), y=full_proc.tolist(), name=f'{FS} Hz', line=dict(width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=t_250.tolist(), y=sig_250.tolist(), name='250 Hz', line=dict(width=1, color='orange')), row=2, col=1)
fig.update_layout(height=500, title='Resampling', template='plotly_white')
fig.show()

## 8. Segment quality check (beat segments)

In [ ]:
beats, beat_peaks = segment_beats(full_proc, FS, window=187)
print(f'Extracted {len(beats)} beats')

if len(beats) > 0:
    fig = go.Figure()
    n_show = min(20, len(beats))
    for i in range(n_show):
        fig.add_trace(go.Scatter(
            x=list(range(187)), y=beats[i].tolist(),
            mode='lines', name=f'beat {i}', line=dict(width=0.8),
            opacity=0.5,
        ))
    fig.add_trace(go.Scatter(
        x=list(range(187)), y=beats[:n_show].mean(axis=0).tolist(),
        mode='lines', name='mean beat', line=dict(width=2, color='red'),
    ))
    fig.update_layout(title='Beat Segments (overlay)', xaxis_title='Sample',
                      template='plotly_white', showlegend=False)
    fig.show()
else:
    print('No beats extracted.')

## 9. Full Preprocessor pipeline

In [ ]:
# Simulate a small batch
X_fake = np.stack([sig] * 4)[:, np.newaxis, :]  # (4, 1, n_t)
prep = Preprocessor(fs=FS, target_fs=250)
X_proc = prep.fit_transform(X_fake)
print(f'Input shape : {X_fake.shape}')
print(f'Output shape: {X_proc.shape}')
print(f'Mean  (should ~0): {X_proc.mean():.4f}')
print(f'Std   (should ~1): {X_proc.std():.4f}')